In [1]:
DATA_PATH = r"../data/parsed/overs.csv"               # FULL output of the updated parse_cricsheet.py
FIRST_INNINGS_PATH = r"../data/parsed/first_innings_states.csv"  # 1st-innings states for empirical DLS curve
OUT_DIR   = r"../data/features"                       # where features_all.csv and the helper files are written

# Used ONLY to keep the test seasons out of the diagnostics at the bottom. It does not split anything.
CHECKS_EXCLUDE_FROM_SEASON = 2025
DLS_FIT_BEFORE_SEASON = 2024   # fit the empirical DLS resource curve strictly on training seasons (< 2024)

# How much to trust small samples: "pseudo-observations" that pull an estimate towards a prior.
K_SEASON = 5            # season average target  -> pulled towards last season's average
RECENT_PAR_N, K_RECENT_PAR = 30, 5  # rolling 30-match par target -> pulled towards 170 when < 30 matches
K_VENUE_PAR = 8         # venue average target   -> pulled towards rolling recent par
K_VENUE  = 10           # venue chase win rate   -> pulled towards the league chase win rate
K_HOME   = 10           # team home record       -> pulled towards the league home win rate
FORM_N, K_FORM = 5, 2   # form = last 5 matches this season, pulled towards 0.5 by 2 pseudo-matches
SR_PRIOR, SR_K = 125.0, 60   # past strike rate -> pulled towards 125 by 60 pseudo-balls
BPD_PRIOR = 18.0        # past balls per dismissal prior
RRR_CAP = 36.0          # more than 6 runs a ball is hopeless anyway, so cap the required rate there

In [2]:
import json, re
from collections import defaultdict, deque
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

df = pd.read_csv(DATA_PATH, dtype={"match_id": str})
fi = pd.read_csv(FIRST_INNINGS_PATH, dtype={"match_id": str})

NEEDED = ["match_id", "date", "venue", "batting_team", "bowling_team", "target", "chasing_team_won",
          "over", "runs_so_far", "wickets_lost", "balls_left", "runs_needed", "current_run_rate",
          "required_run_rate", "runs_last_3", "wkts_last_3",
          # parse columns added by the updated parser
          "striker_id", "non_striker_id", "striker_past_sr", "non_striker_past_sr",
          "striker_past_balls", "non_striker_past_balls", "striker_balls_faced", "non_striker_balls_faced",
          "striker_runs", "non_striker_runs", "striker_past_bpd", "non_striker_past_bpd",
          "rem_bat_n", "rem_bat_quality", "rem_bat_sr",
          "bowl_rem_econ", "bowl_rem_wkt_rate", "best_bowler_overs_left",
          "chasing_impact_used", "bowling_impact_used", "boundaries_last_3", "balls_since_last_wicket",
          "chasing_team_home", "bowling_team_home"]
missing = [c for c in NEEDED if c not in df.columns]
assert not missing, f"Columns missing: {missing}. Re-run the UPDATED parse_cricsheet.py on the full folder."

# Dates: the parser writes ISO (2017-04-05); a file re-saved in Excel becomes 05-04-17 (day-month-year).
s = df["date"].astype(str)
if s.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all():
    df["date"] = pd.to_datetime(s, format="%Y-%m-%d")
elif s.str.fullmatch(r"\d{2}-\d{2}-\d{2}").all():
    print("Dates look Excel-formatted (dd-mm-yy); parsing them that way.")
    df["date"] = pd.to_datetime(s, format="%d-%m-%y")
else:
    raise ValueError("Unrecognised date format in the date column.")

df["season"] = df["date"].dt.year          # IPL edition = calendar year
df = df.sort_values(["date", "match_id", "over"]).reset_index(drop=True)

print(f"{df.match_id.nunique()} matches, {len(df)} rows, seasons {df.season.min()}-{df.season.max()}")
print(df.groupby("season").agg(matches=("match_id", "nunique"), rows=("over", "size")).T.to_string())

1185 matches, 21251 rows, seasons 2008-2026
season   2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
matches    54    53    59    69    73    73    57    52    56    57    57    57    56    59    74    71    70    68    70
rows      938   963  1064  1220  1332  1327  1032   939  1001   993  1040  1040  1012  1052  1332  1293  1239  1202  1232


In [3]:
VENUE_RULES = [  # (fragment of the lower-case raw name, one clean name)
    ("chinnaswamy", "M Chinnaswamy Stadium, Bengaluru"), ("chidambaram", "MA Chidambaram Stadium, Chennai"),
    ("wankhede", "Wankhede Stadium, Mumbai"), ("brabourne", "Brabourne Stadium, Mumbai"),
    ("dy patil", "DY Patil Stadium, Navi Mumbai"), ("eden gardens", "Eden Gardens, Kolkata"),
    ("arun jaitley", "Arun Jaitley Stadium, Delhi"), ("feroz shah", "Arun Jaitley Stadium, Delhi"),
    ("narendra modi", "Narendra Modi Stadium, Ahmedabad"), ("sardar patel", "Narendra Modi Stadium, Ahmedabad"),
    ("motera", "Narendra Modi Stadium, Ahmedabad"), ("rajiv gandhi", "Rajiv Gandhi Stadium, Hyderabad"),
    ("sawai mansingh", "Sawai Mansingh Stadium, Jaipur"), ("punjab cricket association", "PCA Stadium, Mohali"),
    ("bindra", "PCA Stadium, Mohali"), ("maharaja yadavindra", "Mullanpur Stadium"),
    ("himachal pradesh", "HPCA Stadium, Dharamsala"), ("maharashtra cricket association", "MCA Stadium, Pune"),
    ("subrata roy", "MCA Stadium, Pune"), ("rajasekhara reddy", "ACA-VDCA Stadium, Visakhapatnam"),
    ("zayed", "Sheikh Zayed Stadium, Abu Dhabi"), ("sharjah", "Sharjah Cricket Stadium"),
    ("dubai international", "Dubai International Stadium"), ("ekana", "Ekana Stadium, Lucknow"),
    ("barsapara", "Barsapara Stadium, Guwahati"), ("saurashtra cricket association", "SCA Stadium, Rajkot"),
    ("green park", "Green Park, Kanpur"), ("holkar", "Holkar Stadium, Indore"), ("jsca", "JSCA Stadium, Ranchi"),
    ("vidarbha", "VCA Stadium, Nagpur"), ("shaheed veer narayan", "SVNS Stadium, Raipur"),
    ("barabati", "Barabati Stadium, Cuttack"), ("nehru stadium", "Nehru Stadium"),
]

def clean_venue(raw):
    low = str(raw).lower()
    for frag, name in VENUE_RULES:
        if frag in low:
            return name
    return str(raw).split(",")[0].strip()   # anything else: name before the first comma

df["venue_clean"] = df["venue"].map(clean_venue)
print(f"{df.venue.nunique()} raw venue names -> {df.venue_clean.nunique()} grounds")
unmapped = sorted({v for v in df.venue.unique() if not any(f in str(v).lower() for f, _ in VENUE_RULES)})
print("Venues that fell through to 'name before the comma' (check these):", unmapped)
(df.groupby("venue_clean").venue.agg(lambda x: " | ".join(sorted(set(x)))).to_frame("raw spellings").head(60))

60 raw venue names -> 36 grounds
Venues that fell through to 'name before the comma' (check these): ['Buffalo Park', 'De Beers Diamond Oval', 'Kingsmead', 'New Wanderers Stadium', 'Newlands', 'OUTsurance Oval', "St George's Park", 'SuperSport Park']


                                                                                                                                                                                                                                        raw spellings
venue_clean                                                                                                                                                                                                                                          
ACA-VDCA Stadium, Visakhapatnam                                                                                              Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium | Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam
Arun Jaitley Stadium, Delhi                                                                                                                                                     Arun Jaitley Stadium | Arun Jaitley Stadium, Delhi | Feroz Shah Kotla
Barabati Stadium

In [4]:
mt = (df.groupby("match_id").agg(
        date=("date", "first"), season=("season", "first"), venue=("venue_clean", "first"),
        chaser=("batting_team", "first"), bowler=("bowling_team", "first"), target=("target", "first"),
        won=("chasing_team_won", "first"), chaser_home=("chasing_team_home", "first"),
        bowler_home=("bowling_team_home", "first"))
      .reset_index().sort_values(["date", "match_id"]).reset_index(drop=True))

def history_features(mt):
    season_sum, season_n = defaultdict(float), defaultdict(int)
    season_mean_final = mt.groupby("season").target.mean().to_dict()   # only ever read for EARLIER seasons
    venue_w, venue_n = defaultdict(int), defaultdict(int)
    venue_t_sum = defaultdict(float)
    recent_targets = deque(maxlen=RECENT_PAR_N)
    league_w = league_n = 0                       # chase wins / matches so far
    home_w, home_n = defaultdict(int), defaultdict(int)   # team -> wins / games when at home
    lh_w = lh_n = 0                               # league-wide: home side wins / games
    recent = defaultdict(lambda: deque(maxlen=FORM_N))    # (team, season) -> last results (1 = win)
    out = []
    for _, day in mt.groupby("date", sort=True):
        for r in day.itertuples():                # 1) features from the past only
            prior = season_mean_final.get(r.season - 1, np.nan)
            n, s = season_n[r.season], season_sum[r.season]
            if np.isnan(prior):
                avg = s / n if n else np.nan
            else:
                avg = (s + K_SEASON * prior) / (n + K_SEASON)
            r_par = (sum(recent_targets) + K_RECENT_PAR * 170.0) / (len(recent_targets) + K_RECENT_PAR)
            v_par = (venue_t_sum[r.venue] + K_VENUE_PAR * r_par) / (venue_n[r.venue] + K_VENUE_PAR)
            p0 = league_w / league_n if league_n else 0.5
            ph = lh_w / lh_n if lh_n else 0.5
            edge = 0.0
            if r.chaser_home:
                edge = (home_w[r.chaser] + K_HOME * ph) / (home_n[r.chaser] + K_HOME) - ph
            elif r.bowler_home:
                edge = -((home_w[r.bowler] + K_HOME * ph) / (home_n[r.bowler] + K_HOME) - ph)
            def form(team):
                h = recent[(team, r.season)]
                return (sum(h) + K_FORM * 0.5) / (len(h) + K_FORM)
            out.append((r.match_id, r.target - avg, r.target - r_par, r.target - v_par,
                        (venue_w[r.venue] + K_VENUE * p0) / (venue_n[r.venue] + K_VENUE),
                        edge, form(r.chaser), form(r.bowler)))
        for r in day.itertuples():                # 2) only now add this day's matches to the past
            season_sum[r.season] += r.target; season_n[r.season] += 1
            venue_w[r.venue] += r.won; venue_n[r.venue] += 1
            venue_t_sum[r.venue] += r.target
            recent_targets.append(r.target)
            league_w += r.won; league_n += 1
            if r.chaser_home:
                home_w[r.chaser] += r.won; home_n[r.chaser] += 1; lh_w += r.won; lh_n += 1
            elif r.bowler_home:
                home_w[r.bowler] += 1 - r.won; home_n[r.bowler] += 1; lh_w += 1 - r.won; lh_n += 1
            recent[(r.chaser, r.season)].append(r.won)
            recent[(r.bowler, r.season)].append(1 - r.won)
    return pd.DataFrame(out, columns=["match_id", "target_vs_season_avg", "target_vs_recent_par",
                                      "target_vs_venue_par", "venue_chase_rate",
                                      "home_edge", "chaser_form", "bowler_form"]).set_index("match_id")

hist = history_features(mt)
hist.describe().round(3)

       target_vs_season_avg  target_vs_recent_par  target_vs_venue_par  venue_chase_rate  home_edge  chaser_form  bowler_form
count              1184.000              1185.000             1185.000          1185.000   1185.000     1185.000     1185.000
mean                  0.227                 0.482                4.810             0.541      0.001        0.505        0.506
std                  31.517                31.467               32.382             0.067      0.071        0.161        0.161
min                -120.544              -120.829             -112.848             0.000     -0.277        0.143        0.143
25%                 -19.185               -19.143              -15.465             0.500     -0.041        0.429        0.429
50%                   0.491                 1.429                4.764             0.543      0.000        0.500        0.500
75%                  20.701                21.143               25.217             0.582      0.041        0.600      

In [5]:
def brute(m):
    past = mt[mt.date < m.date]
    ps = past[past.season == m.season]
    prior = mt.loc[mt.season == m.season - 1, "target"].mean()
    n, s = len(ps), ps.target.sum()
    if np.isnan(prior):
        avg = s / n if n else np.nan
    else:
        avg = (s + K_SEASON * prior) / (n + K_SEASON)
    p0 = past.won.mean() if len(past) else 0.5
    rec_t = past.target.tail(RECENT_PAR_N)
    r_par = (rec_t.sum() + K_RECENT_PAR * 170.0) / (len(rec_t) + K_RECENT_PAR)
    pv = past[past.venue == m.venue]
    v_par = (pv.target.sum() + K_VENUE_PAR * r_par) / (len(pv) + K_VENUE_PAR)
    vcr = (pv.won.sum() + K_VENUE * p0) / (len(pv) + K_VENUE)
    # home record: games where a team was the home side, and whether it won
    hg = pd.concat([
        past[past.chaser_home == 1].assign(team=lambda x: x.chaser, hw=lambda x: x.won),
        past[(past.bowler_home == 1) & (past.chaser_home != 1)].assign(team=lambda x: x.bowler, hw=lambda x: 1 - x.won)])
    ph = hg.hw.mean() if len(hg) else 0.5
    def home_rec(team):
        g = hg[hg.team == team]
        return (g.hw.sum() + K_HOME * ph) / (len(g) + K_HOME) - ph
    edge = home_rec(m.chaser) if m.chaser_home else (-home_rec(m.bowler) if m.bowler_home else 0.0)
    def form(team):
        g = ps[(ps.chaser == team) | (ps.bowler == team)].tail(FORM_N)
        res = np.where(g.chaser == team, g.won, 1 - g.won)
        return (res.sum() + K_FORM * 0.5) / (len(g) + K_FORM)
    return [m.target - avg, m.target - r_par, m.target - v_par, vcr, edge, form(m.chaser), form(m.bowler)]

sample = mt.sample(60, random_state=0)
bad = 0
for m in sample.itertuples():
    exp = np.array(brute(m), dtype=float)
    got = hist.loc[m.match_id].to_numpy(dtype=float)
    ok = np.all((np.isnan(exp) & np.isnan(got)) | np.isclose(exp, got, atol=1e-9))
    bad += (not ok)
print(f"matches checked: {len(sample)} | disagreements: {bad}")
assert bad == 0, "history features disagree with the brute-force version - do not use them"

matches checked: 60 | disagreements: 0


In [6]:
d = df.copy()
d["wickets_in_hand"] = 10 - d.wickets_lost                      # internal helper, not a model feature
d["required_run_rate"] = d.required_run_rate.clip(upper=RRR_CAP)
d["run_rate_diff"] = d.required_run_rate - d.current_run_rate
d["runs_needed_frac"] = d.runs_needed / d.target
d["resources"] = d.balls_left * d.wickets_in_hand                # kept in META for backwards compatibility
d["runs_needed_per_wicket"] = d.runs_needed / d.wickets_in_hand
d["acceleration"] = d.runs_last_3 / d.over.clip(upper=3) - d.current_run_rate

# Fit empirical DLS exponential resource curve R(balls_left, wickets_lost) strictly on training 1st innings
fi_tr = fi[fi.season < DLS_FIT_BEFORE_SEASON]
dls_Z0, dls_b = {}, {}
for w in range(10):
    sub = fi_tr[fi_tr.wickets_lost == w]
    if len(sub) >= 30:
        grp = sub.groupby("balls_left").runs_remaining.mean()
        u_vals, y_vals = grp.index.to_numpy(dtype=float), grp.to_numpy(dtype=float)
        p0 = [max(20.0, 175.0 * ((10 - w) / 10.0) ** 1.3), 0.012 + 0.003 * w]
        try:
            popt, _ = curve_fit(lambda u, Z0, b: Z0 * (1.0 - np.exp(-b * u)), u_vals, y_vals,
                                p0=p0, bounds=([5.0, 0.002], [400.0, 0.25]), maxfev=5000)
            dls_Z0[w], dls_b[w] = float(popt[0]), float(popt[1])
        except Exception:
            dls_Z0[w], dls_b[w] = p0[0], p0[1]
    else:
        dls_Z0[w], dls_b[w] = max(12.0, 175.0 * ((10 - w) / 10.0) ** 1.4), 0.015 + 0.004 * w
# Enforce monotone decreasing asymptotic runs Z0(w) as wickets fall
for w in range(1, 10):
    dls_Z0[w] = min(dls_Z0[w], dls_Z0[w - 1] * 0.92)
norm_120 = dls_Z0[0] * (1.0 - np.exp(-dls_b[0] * 120.0))
w_arr = d.wickets_lost.clip(0, 9).to_numpy()
z0_vec = np.array([dls_Z0[w] for w in w_arr])
b_vec = np.array([dls_b[w] for w in w_arr])
d["dls_resources"] = np.clip(z0_vec * (1.0 - np.exp(-b_vec * d.balls_left.to_numpy())) / norm_120 * 100.0, 0.0, 100.0)
d["par_score_diff"] = d.runs_so_far - d.target * (1.0 - d["dls_resources"] / 100.0)

# Match phase & Powerplay indicators
d["is_powerplay"] = (d.over <= 6).astype(int)
d["is_death"] = (d.over >= 15).astype(int)
d["powerplay_balls_left"] = np.maximum(0, 36 - (120 - d.balls_left))

def shrunk_sr(sr, balls):
    balls = balls.fillna(0)
    sr = sr.fillna(SR_PRIOR)
    return (sr * balls + SR_PRIOR * SR_K) / (balls + SR_K)

d["bat_sr_mean"] = (shrunk_sr(d.striker_past_sr, d.striker_past_balls)
                    + shrunk_sr(d.non_striker_past_sr, d.non_striker_past_balls)) / 2
d["bat_bpd_mean"] = (d.striker_past_bpd.fillna(BPD_PRIOR) + d.non_striker_past_bpd.fillna(BPD_PRIOR)) / 2
d["bat_balls_sum"] = d.striker_balls_faced + d.non_striker_balls_faced
d["bat_balls_max"] = np.maximum(d.striker_balls_faced, d.non_striker_balls_faced)
d["bat_runs_sum"] = d.striker_runs.fillna(0) + d.non_striker_runs.fillna(0)
d["settled_sr_interaction"] = (d.bat_balls_max * d.bat_sr_mean) / 100.0
no_batters = d.striker_id.isna() | d.non_striker_id.isna()      # no next over -> batters unknown
d.loc[no_batters, ["bat_sr_mean", "bat_bpd_mean", "bat_balls_sum", "bat_balls_max", "bat_runs_sum", "settled_sr_interaction"]] = np.nan

d = d.merge(hist, left_on="match_id", right_index=True, how="left")
print("rows:", len(d), "| rows without known batters:", int(no_batters.sum()))

rows: 21251 | rows without known batters: 6


In [7]:
GROUPS = {
    "core":     ["runs_so_far", "target_vs_venue_par", "wickets_lost", "balls_left", "current_run_rate", "required_run_rate"],
    "derived":  ["run_rate_diff", "runs_needed_frac", "dls_resources", "par_score_diff", "runs_needed_per_wicket"],
    "phase":    ["is_powerplay", "is_death", "powerplay_balls_left"],
    "momentum": ["runs_last_3", "wkts_last_3", "boundaries_last_3", "acceleration", "balls_since_last_wicket"],
    "batters":  ["bat_sr_mean", "bat_bpd_mean", "bat_balls_max", "bat_runs_sum", "settled_sr_interaction", "rem_bat_quality", "rem_bat_sr"],
    "bowling":  ["bowl_rem_econ", "bowl_rem_wkt_rate", "best_bowler_overs_left"],
    "impact":   ["chasing_impact_used", "bowling_impact_used"],
    "history":  ["target_vs_season_avg", "target_vs_recent_par", "venue_chase_rate"],
    "home":     ["chasing_team_home", "bowling_team_home", "home_edge"],
    "form":     ["chaser_form", "bowler_form"],
}
FEATURES = [f for g in GROUPS.values() for f in g]
assert len(FEATURES) == len(set(FEATURES)) and all(f in d.columns for f in FEATURES)

DEFS = {
    "runs_so_far": "Current score of the chase",
    "target_vs_venue_par": "Target minus the past-only venue average target shrunk towards recent league par",
    "wickets_lost": "Wickets down",
    "balls_left": "Legal balls remaining (wides and no-balls do not count)",
    "current_run_rate": "Runs per over so far",
    "required_run_rate": "Runs needed per over for the rest of the chase, capped at 36",
    "run_rate_diff": "Required minus current run rate",
    "runs_needed_frac": "Runs needed / target",
    "dls_resources": "Empirical DLS resource percentage remaining (0-100), fitted on training 1st innings",
    "par_score_diff": "Runs so far minus the empirical DLS par score for the current balls left and wickets lost",
    "runs_needed_per_wicket": "Runs needed / wickets in hand",
    "is_powerplay": "1 during powerplay overs (overs 1-6), 0 otherwise",
    "is_death": "1 during death overs (overs 15-19), 0 otherwise",
    "powerplay_balls_left": "Legal powerplay balls remaining in the innings (0 after over 6)",
    "runs_last_3": "Runs scored in the last 3 completed overs",
    "wkts_last_3": "Wickets lost in the last 3 completed overs",
    "boundaries_last_3": "Fours and sixes off the bat in the last 3 overs",
    "acceleration": "Run rate over the last 3 overs minus the overall current run rate",
    "balls_since_last_wicket": "Legal balls since the last wicket (current partnership)",
    "bat_sr_mean": "Mean past IPL strike rate of the two batters at the crease, pulled towards 125 when history is short",
    "bat_bpd_mean": "Mean past IPL balls-per-dismissal of the two batters at the crease, pulled towards 18",
    "bat_balls_max": "Balls faced this innings by the more settled of the two batters",
    "bat_runs_sum": "Combined runs scored this innings by the two batters currently at the crease",
    "settled_sr_interaction": "Interaction of set batter balls faced and crease strike rate (bat_balls_max * bat_sr_mean / 100)",
    "rem_bat_quality": "Sum of top-3 shrunk past batting averages among un-batted players in the chasing squad",
    "rem_bat_sr": "Mean shrunk past strike rate of the top-3 un-batted players in the chasing squad",
    "bowl_rem_econ": "Expected economy rate of remaining overs using best available bowlers within 4-over quotas",
    "bowl_rem_wkt_rate": "Expected wickets per ball of remaining overs using best available bowlers within 4-over quotas",
    "best_bowler_overs_left": "Combined overs remaining (0-8) for the defending side's two best bowlers",
    "chasing_impact_used": "1 if the chasing team has made its Impact Player substitution",
    "bowling_impact_used": "1 if the bowling team has made its Impact Player substitution",
    "target_vs_season_avg": "Target minus the average target of this season's earlier matches (past only)",
    "target_vs_recent_par": "Target minus the rolling 30-match IPL average target before this date (past only)",
    "venue_chase_rate": "Past chase win rate at the ground, pulled towards the league rate (past only)",
    "chasing_team_home": "1 if the chasing team is playing at one of its home grounds",
    "bowling_team_home": "1 if the bowling team is playing at one of its home grounds",
    "home_edge": "Home side's past home record minus league home win rate; positive favours the chasing team; 0 if neutral",
    "chaser_form": "Chasing team's win rate in its last 5 matches this season (past only, shrunk to 0.5)",
    "bowler_form": "Bowling team's win rate in its last 5 matches this season (past only, shrunk to 0.5)",
}
dictionary = pd.DataFrame([(f, g, DEFS[f]) for g, fs in GROUPS.items() for f in fs],
                          columns=["feature", "group", "definition"])
dictionary

                    feature     group                                                                                                definition
0               runs_so_far      core                                                                                Current score of the chase
1       target_vs_venue_par      core                          Target minus the past-only venue average target shrunk towards recent league par
2              wickets_lost      core                                                                                              Wickets down
3                balls_left      core                                                   Legal balls remaining (wides and no-balls do not count)
4          current_run_rate      core                                                                                      Runs per over so far
5         required_run_rate      core                                              Runs needed per over for the rest of the chase, cappe

In [8]:
META = ["match_id", "date", "season", "over", "batting_team", "bowling_team", "venue_clean", "target", "resources", "bat_balls_sum", "chasing_team_won"]
out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)
d[META + FEATURES].to_csv(out / "features_all.csv", index=False)
(out / "feature_groups.json").write_text(json.dumps(GROUPS, indent=2))
(out / "dls_params.json").write_text(json.dumps({"Z0": {str(k): round(v, 4) for k, v in dls_Z0.items()},
                                                 "b": {str(k): round(v, 6) for k, v in dls_b.items()},
                                                 "norm_120": round(float(norm_120), 4)}, indent=2))
dictionary.to_csv(out / "feature_dictionary.csv", index=False)
print("saved to", out.resolve())
print(f"features_all.csv: {d.match_id.nunique()} matches, {len(d)} rows, {len(FEATURES)} features")

saved to C:\Users\LEGION\OneDrive\Documents\GitHub\es335-2026\data\features
features_all.csv: 1185 matches, 21251 rows, 39 features


In [9]:
chk = d[d.season < CHECKS_EXCLUDE_FROM_SEASON]
print(f"Diagnostics use {chk.match_id.nunique()} matches / {len(chk)} rows from seasons before {CHECKS_EXCLUDE_FROM_SEASON}.")
checks = pd.DataFrame({
    "missing": chk[FEATURES].isna().sum(),
    "infinite": np.isinf(chk[FEATURES]).sum(),
    "mean": chk[FEATURES].mean().round(3),
    "std": chk[FEATURES].std().round(3),
    "min": chk[FEATURES].min().round(3),
    "max": chk[FEATURES].max().round(3),
})
display(checks)

corr = chk[FEATURES].corr().abs()
pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
         .loc[lambda s: s > 0.9].sort_values(ascending=False).round(3))
print("Pairs with |correlation| > 0.9:")
print(pairs.to_string() if len(pairs) else "none")

Diagnostics use 1047 matches / 18817 rows from seasons before 2025.
Pairs with |correlation| > 0.9:
bat_balls_max         settled_sr_interaction    0.993
target_vs_season_avg  target_vs_recent_par      0.989
balls_left            dls_resources             0.972
target_vs_venue_par   target_vs_recent_par      0.970
                      target_vs_season_avg      0.959
runs_so_far           runs_needed_frac          0.940
balls_left            runs_needed_frac          0.933
runs_so_far           balls_left                0.930
required_run_rate     run_rate_diff             0.925


                         missing  infinite     mean     std      min      max
runs_so_far                    0         0   74.753  45.588    0.000  253.000
target_vs_venue_par            0         0    4.393  30.042 -112.848  113.680
wickets_lost                   0         0    2.482   2.088    0.000    9.000
balls_left                     0         0   62.468  31.919    3.000  114.000
current_run_rate               0         0    7.654   2.126    0.000   26.000
required_run_rate              0         0    9.951   5.054    0.077   36.000
run_rate_diff                  0         0    2.297   5.600  -19.474   31.368
runs_needed_frac               0         0    0.553   0.264    0.005    1.000
dls_resources                  0         0   55.139  27.105    2.452   97.225
par_score_diff                 0         0   -1.832  23.331 -148.724  104.481
runs_needed_per_wicket         0         0   13.261   8.928    0.100  162.000
is_powerplay                   0         0    0.334   0.472    0